# CodeDoctor — GPU Acceleration Addendum (Google Colab T4)

This notebook runs the **offline evaluation, full-corpus ablation (T6), recall ceiling (T3), and MTEB benchmark generation** with GPU hardware acceleration (NVIDIA T4 16GB).

- **Authority:** `THEME1-BUILD-PLAN-v3.pdf` & `changes_3.md`.
- **Model:** `codefuse-ai/F2LLM-v2-0.6B` (inference only, no fine-tuning).
- **Constraint:** Live service and juror re-indexing remain strictly CPU-based.

In [ ]:
# 1. Verify GPU environment
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Please ensure a T4 GPU is selected in the kernel / Colab settings.")


In [ ]:
# 2. Workspace Setup & Dependency Installation
import os, base64, io, zipfile
from pathlib import Path

if not Path("app").exists():
    print("Remote Colab environment detected: unpacking CodeDoctor workspace bundle...")
    _BUNDLE_B64 = "..."
    zipfile.ZipFile(io.BytesIO(base64.b64decode(_BUNDLE_B64))).extractall(".")
    print("CodeDoctor workspace extracted successfully!")
else:
    print("Local CodeDoctor workspace detected.")

!pip install -q mteb sentence-transformers accelerate faiss-cpu datasets pyyaml tree-sitter tree-sitter-python tree-sitter-javascript tree-sitter-typescript scipy


In [ ]:
# 3. Run Throughput Benchmark on T4 GPU (Task 3.1 & 3.5)
# Measures CPU baseline + B5 latency, then measures T4 GPU throughput & speedup
!python scripts/benchmark_throughput.py --device auto


In [ ]:
# 4. Plan T3: Recall Ceiling Measurement on Real Split (Task 3.6)
# Measures recall@10, @50, @100, @200 to establish the heavy-reranker ceiling
!python scripts/measure_recall_ceiling.py --max-queries 50 --batch-size 256 --device cuda


In [ ]:
# 5. Plan T6: Full-Corpus Ablation Matrix on T4 GPU (Task 3.3)
# Evaluates base vs MiniLM, prefix vs no-prefix, and dense vs hybrid with bootstrap CIs
!python scripts/run_ablation.py --max-queries 150 --max-corpus 0 --batch-size 256 --device cuda


In [ ]:
# 6. Official MTEB AppsRetrieval Submission Evaluation (Task 2.3)
# Produces official submission artifact: appsretrieval_results.json
!python scripts/evaluate_mteb.py --output appsretrieval_results.json --batch-size 256 --device cuda


In [ ]:
# 7. Verify artifacts and display results
import json
from pathlib import Path

res_path = Path("appsretrieval_results.json")
if res_path.exists():
    with open(res_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("\n=== MTEB AppsRetrieval Official Scores ===")
    for split, split_data in data.get("scores", {}).items():
        print(f"\nSplit [{split}]:")
        items = split_data if isinstance(split_data, list) else [split_data]
        for entry in items:
            if isinstance(entry, dict):
                for k, v in entry.items():
                    if isinstance(v, float):
                        print(f"  {k}: {v:.4f}")
                    elif not isinstance(v, (dict, list)):
                        print(f"  {k}: {v}")
else:
    print("appsretrieval_results.json not found yet.")
